<div style="border-left: 5px solid #b7791f; background-color: #fff8e1; padding: 0.8em 1em; margin: 1em 0; border-radius: 4px;">
  <strong>Warning: AI-assisted materials</strong><br><br>
  These materials were developed with assistance from AI tools. All content has been reviewed and edited by the instructor, who takes final responsibility for its accuracy, clarity, and appropriateness for the course. Students should treat these materials as instructor-reviewed course content while applying the same critical judgment they would use with any technical material. Please report any suspected errors or unclear explanations to ghunt@wm.edu.
</div>

# Random Forests and Bagging

In the previous lecture, we studied trees: classification and regression trees. A single tree is easy to interpret and can capture nonlinear structure, but it is often unstable. A small change in the training data can produce a noticeably different tree.

Random forests are built around one main idea: average many randomized trees. We'll see that this allows the variance to go down while the bias stays relatively low.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.datasets import make_moons, make_regression, load_diabetes, load_breast_cancer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, accuracy_score, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(657677)

## Recap: a single tree

A tree partitions the input space into leaf regions

$$
R_1,\dots,R_M.
$$

For regression, a tree has score function

$$
\hat s(x)=\sum_{m=1}^M \hat c_m \mathbf{1}\{x\in R_m\},
$$

where

$$
\hat c_m
=\frac{1}{N_m}\sum_{n:x_n\in R_m}y_n.
$$

For classification, each leaf has a vector of class proportions

$$
\hat p_m=\begin{bmatrix}
\hat p_{m1}\\
\vdots\\
\hat p_{mK}
\end{bmatrix},
\qquad
\hat p_{mk}
=\frac{1}{N_m}\sum_{n:x_n\in R_m}\mathbf{1}\{y_n=k\}.
$$

The score function is

$$
\hat s(x)
=\sum_{m=1}^M \hat p_m \mathbf{1}\{x\in R_m\}.
$$

Then

$$
\hat f(x)
=\arg\max_k \hat s_k(x).
$$

A single deep tree usually has low bias but high variance.

## Expectation and variance of averages

Before bagging, we need one basic fact: averaging reduces variance.

Let

$$
Z_1,\dots,Z_B
$$

be random variables with the same mean and variance:

$$
\mathbb{E}[Z_b]=\mu,
\qquad
\operatorname{Var}(Z_b)=\sigma^2.
$$

Define their average

$$
\bar Z_B
=\frac{1}{B}\sum_{b=1}^B Z_b.
$$

Then

$$
\mathbb{E}[\bar Z_B]
=\mu.
$$

So averaging does not change the mean.


If $Z_1,\dots,Z_B$ are uncorrelated, then

$$
\operatorname{Var}(\bar Z_B)
=\operatorname{Var}\left(\frac{1}{B}\sum_{b=1}^B Z_b\right)
=\frac{1}{B^2}\sum_{b=1}^B \operatorname{Var}(Z_b).
$$

Since each variance is $\sigma^2$,

$$
\operatorname{Var}(\bar Z_B)
=\frac{1}{B^2}B\sigma^2
=\frac{\sigma^2}{B}.
$$

So if the $Z_b$ are independent, averaging $B$ of them reduces variance by a factor of $B$.

In [ ]:
# Simulate the variance reduction from averaging independent estimators.

B_values = np.arange(1, 101)
n_reps = 5000

average_variances = []

for B in B_values:
    Z = rng.exponential(size=(n_reps, B))
    Z_bar = Z.mean(axis=1)
    average_variances.append(np.var(Z_bar, ddof=1))

plt.figure(figsize=(6, 4))
plt.plot(B_values, average_variances, label="simulated variance")
plt.plot(B_values, 1 / B_values, linestyle="--", label="$1/B$")
plt.xlabel("number of estimators $B$")
plt.ylabel(r"$\operatorname{Var}(\bar Z_B)$")
plt.title("Variance reduction from averaging independent estimators")
plt.legend()
plt.tight_layout()
plt.show()

Now, if the $Z_b$ aren't independent, we'll need to slightly modify things. Assume

$$
\operatorname{Var}(Z_b)=\sigma^2
$$

and every pair has the same correlation

$$
\operatorname{Corr}(Z_b,Z_{b'})=\rho
\qquad
b\neq b'.
$$

Then

$$
\operatorname{Cov}(Z_b,Z_{b'})=\rho\sigma^2.
$$

Therefore,

$$
\operatorname{Var}(\bar Z_B)
=\frac{1}{B^2}
\left[
B\sigma^2 + B(B-1)\rho\sigma^2
\right].
$$

Simplifying,

$$
\operatorname{Var}(\bar Z_B)
=\rho\sigma^2
+
\frac{1-\rho}{B}\sigma^2.
$$

Consequently:

- If $\rho=0$, then $\operatorname{Var}(\bar Z_B)=\sigma^2/B$.
- If $\rho=1$, then $\operatorname{Var}(\bar Z_B)=\sigma^2$.
- If $0<\rho<1$, averaging helps, but the variance cannot go below $\rho\sigma^2$.

So averaging reduces variance most when the observations not too correlated.

In [ ]:
# Variance of an average as a function of B for several correlations.

B_values = np.arange(1, 301)
rho_values = [0.0, 0.05, 0.2, 0.5, 0.9, 1]
sigma2 = 1.0

plt.figure(figsize=(7, 4))

for rho in rho_values:
    var_avg = rho * sigma2 + (1 - rho) * sigma2 / B_values
    plt.plot(B_values, var_avg, label=fr"$\rho={rho}$")

plt.xlabel("number of estimators $B$")
plt.ylabel(r"$\operatorname{Var}(\bar Z_B)$")
plt.title("Averaging helps less when estimators are correlated")
plt.legend()
plt.tight_layout()
plt.show()

## Bagging: bootstrap aggregation

Using these properties of averaging we can make regression and classifiers better by combining, in what's called an ensemble, many individual regression methods or classifiers. This is called bagging. Bagging (stands for **bootstrap aggregation**) whereby we:

1. draw many (**bootstrap**) sub-samples from the training data;
2. fit one tree to each bootstrap sample;
3. combine the resulting predictions.

A **bootstrap sample** from data of size $N$, is a sample of size $N$ that is drawn **with replacement** from the original training data. So each bootstrap sample contains some training observations multiple times and leaves some observations out. The main idea is that a boostrap sample mimics the sampling process:

- real sample: population $\to$ sample
- boostrap sample: sample $\to$ subsample

### Step 1

Let the original training data be

$$
\mathcal{D}_N = \{(x_n,y_n)\}_{n=1}^N.
$$

For $b=1,\dots,B$, draw a bootstrap sample

$$
\mathcal{D}_N^{(b)}
$$

by sampling $N$ training pairs from $\mathcal{D}_N$ with replacement.

### Step 2

Then fit a **base method** to each bootstrap sample:

$$
\hat s_b.
$$

### Step 3

Then we'll combine the various $\hat s_b$. 

#### Regression

For **regression**, each $\hat s$ gives a scalar score

$$
\hat s_b(x) \in \mathbb{R}.
$$

The bagged score function is the average:

$$
\hat s_{\text{bag}}(x)
=\frac{1}{B}\sum_{b=1}^B \hat s_b(x).
$$

Since regression uses the identity action,

$$
\hat f_{\text{bag}}(x)
=\hat s_{\text{bag}}(x).
$$

#### Classification

For bagged **classification**, there are two closely related ways to combine the base methods: **hard voting** and **soft voting**. 

In both cases, each base method $\hat s_b$ produces a "soft" score vector

$$
\hat s_b^\text{soft}(x) \in \mathbb{R}^K
$$
which is our traditional score from a classification tree such that the elements are positive and sum to one. There is the associated $\hat f_b(x) = a(\hat s_b^\text{soft}(x)) \in \{1,\ldots,K\}$. 

In the **hard-vote version**, we can encode this prediction as a one-hot score vector

$$
\hat s_b(x)
=e_{\hat f_b(x)}
\in \mathbb{R}^K,
$$

where $e_k$ is the $k$th standard basis vector, e.g. $(1, 0, 0, \ldots)$, or $(0, 1, 0, \ldots)$. Equivalently,

$$
\hat s_{b,k}(x)=
\mathbf{1}\{\hat f_b(x)=k\}.
$$

The bagged score vector is the average of these one-hot vectors:

$$
\hat s(x)=
\frac{1}{B}
\sum_{b=1}^B
\hat s_b(x).
$$

Thus the $k$th entry is

$$
\hat s_{k}(x)=
\frac{1}{B}
\sum_{b=1}^B
\mathbf{1}\{\hat f_b(x)=k\}.
$$

So $\hat s_{k}(x)$ is the proportion of trees that voted for class $k$. The final prediction is

$$
\hat f(x)=
a(\hat s(x))=
\arg\max_{k \in \{1,\dots,K\}}
\hat s_{k}(x).
$$


The second version is the **soft-vote** version. This just directly averages class-probability vectors:

$$
\hat s(x)
=\frac{1}{B}
\sum_{b=1}^B
\hat s_b^\text{soft}(x),
$$

The final class prediction is still

$$
\hat f(x)
=\arg\max_k
\hat s_{k}(x).
$$

Classical random forests are often described in the first way: hard-voting. Some software implementations, including `sklearn`, use the soft-voting version (see: [link](https://scikit-learn.org/stable/modules/ensemble.html#forest) "In contrast to the original publication, the scikit-learn implementation combines classifiers by averaging their probabilistic prediction, instead of letting each classifier vote for a single class.")

**Tradeoff**. Hard voting is simple and robust. Each tree only contributes its predicted class, so the forest does not rely on the tree's estimated probabilities being very accurate. The cost is that hard voting throws away information. A tree that is barely confident and a tree that is very confident count the same.

Soft voting averages class-probability vectors instead of hard class labels. This uses more information from each tree, since a prediction like

$$
(0.51, 0.49)
$$

is treated differently from

$$
(0.99, 0.01).
$$

The cost is that soft voting depends more on the quality of the probability estimates produced by the individual trees.

When we say probabilities are **calibrated**, we mean that predicted probabilities match empirical frequencies. For example, among observations where a model predicts class 1 with probability about $0.8$, roughly 80% of those observations should actually belong to class 1.

So the tradeoff is:

- hard voting is less sensitive to bad probability estimates, but loses confidence information;
- soft voting uses confidence information, but can be misleading if the probabilities are poorly calibrated.

### Out-of-bag samples

The probability that a fixed training observation is not selected in one draw is

$$
1-\frac{1}{N}.
$$

The probability that it is not selected in any of the $N$ bootstrap draws is

$$
\left(1-\frac{1}{N}\right)^N
\approx e^{-1}
\approx 0.368.
$$

So each bootstrap sample leaves out about $36.8\%$ of the training observations. These left-out observations are called **out-of-bag** observations for that base method. We'll revisit these later. 

In [ ]:
# How much of the original data appears in a bootstrap sample?

N = 1000
n_reps = 5000
unique_fracs = []
oob_fracs = []

for _ in range(n_reps):
    sample = rng.integers(0, N, size=N)
    unique = np.unique(sample)
    unique_fracs.append(len(unique) / N)
    oob_fracs.append(1 - len(unique) / N)

print("Average fraction appearing at least once:", np.mean(unique_fracs).round(4))
print("Average out-of-bag fraction:", np.mean(oob_fracs).round(4))
print("Theoretical in-bag fraction:", (1 - np.exp(-1)).round(4))
print("Theoretical out-of-bag fraction:", np.exp(-1).round(4))

## Bagged regression trees

The advantage of bagging is just like the advantage of averaging individual samples in the case of normal random variables. We can reduce variance while leaving the expected value, i.e. bias, unchanged.

Bagging is especially useful for trees because deep trees are unstable. The individual trees have high variance (but low bias), and averaging can reduce that variance.

In [ ]:
# Simulate one-dimensional regression data.

rng = np.random.default_rng(657677)
N = 250

x = np.sort(rng.uniform(0, 1, size=N))


def true_function(x):
    return (
        1.5 * np.sin(2 * np.pi * x)
        + 1.2 * (x > 0.45)
        - 0.8 * (x > 0.75)
    )

f_true = true_function(x)
y = f_true + rng.normal(scale=0.35, size=N)

X_reg = x.reshape(-1, 1)
x_grid = np.linspace(0, 1, 600)
X_grid = x_grid.reshape(-1, 1)
f_grid = true_function(x_grid)

plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=20, alpha=0.5, label="training data")
plt.plot(x_grid, f_grid, linewidth=2, label="true function")
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.title("Regression data")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compare a single deep tree to a bagged collection of deep trees.

single_tree = DecisionTreeRegressor(
    max_depth=None,
    min_samples_leaf=5,
    random_state=657677
)

bagged_trees = RandomForestRegressor(
    n_estimators=3000, # how many trees to bag: B
    max_features=1.0, # to discuss later
    bootstrap=True,
    min_samples_leaf=5,
    random_state=654654
)

In [ ]:
single_tree.fit(X_reg, y)
bagged_trees.fit(X_reg, y)

y_single = single_tree.predict(X_grid)
y_bag = bagged_trees.predict(X_grid)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.scatter(x, y, s=18, alpha=0.4, label="training data")
plt.plot(x_grid, f_grid, linewidth=2, label="true function")
plt.plot(x_grid, y_single, linewidth=2, label="single tree")
plt.plot(x_grid, y_bag, linewidth=2.5, label="bagged trees")
plt.xlabel("$x$")
plt.ylabel("$y$")
plt.title("Bagging smooths the instability of a single tree")
plt.legend()
plt.tight_layout()
plt.show()

A single tree can jump sharply because one split changes the prediction for an entire region. A bagged predictor averages many such trees, so the final prediction is more stable.

Bagging reduces variance by averaging many unstable trees, but it is not a complete substitute for regularization. If each tree is extremely deep and leaves contain only a few observations, the averaged model can still track noise. Parameters such as `min_samples_leaf`, `max_depth`, and `max_samples` control how flexible the individual trees are.

Bagging helps by averaging many unstable trees, but it helps most when the trees are not too highly correlated. In a one-dimensional example, all trees split on the same predictor, so the bootstrap trees can still be quite correlated. As a result, bagging may smooth the fitted function somewhat, but it may not eliminate overfitting if the individual trees are very deep. Increasing `n_estimators` reduces Monte Carlo noise in the average, but it does not regularize the individual trees. 

Recall, the variance of the average prediction is

$$
\operatorname{Var}
\left(
\frac{1}{B}
\sum_{b=1}^B
\hat s_b(x)
\right)=
\rho \sigma^2
+
\frac{1-\rho}{B}\sigma^2.
$$

As

$$
B \to \infty,
$$

the second term goes to zero, so

$$
\operatorname{Var}
\left(
\frac{1}{B}
\sum_{b=1}^B
\hat s_b(x)
\right)
\to
\rho \sigma^2.
$$

Thus bagging removes the uncorrelated part of the tree variance, but it cannot remove the part shared across trees. If the trees are highly correlated, meaning $\rho$ is large, then the variance reduction from averaging has a limit. Making the individual trees less regularized usually increases the variance of each tree. It may also reduce the correlation between trees, because each tree becomes more sensitive to its bootstrap sample. Bagging benefits from lower correlation, but the relevant large-$B$ quantity is approximately

$$
\rho \sigma^2.
$$

So less regularization is not automatically better. It may lower $\rho$, but it can increase $\sigma^2$ enough that the averaged predictor still looks noisy.

## Bagged classification trees


Bagging classifiers works well for similar, but less easy to caputure mathematically, reasons.

In [ ]:
# Helper for plotting classification decision regions.

def plot_classifier_boundary(model, X, y, title=None, grid_size=250):
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5

    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, grid_size),
        np.linspace(x2_min, x2_max, grid_size)
    )

    grid = np.column_stack([xx1.ravel(), xx2.ravel()])
    pred = model.predict(grid).reshape(xx1.shape)

    plt.figure(figsize=(6, 5))
    plt.contourf(xx1, xx2, pred, alpha=0.25)
    plt.scatter(X[:, 0], X[:, 1], c=y, s=30, alpha=0.8, edgecolor="black", linewidth=0.3)
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    if title is not None:
        plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Simulated classification data.

X_clf, y_clf = make_moons(
    n_samples=500,
    noise=0.28,
    random_state=657677
)

In [ ]:
single_clf_tree = DecisionTreeClassifier(
    max_depth=None,
    min_samples_leaf=2,
    random_state=657677
)

bagged_clf_trees = RandomForestClassifier(
    n_estimators=3000,
    max_features=1.0,      # all features, so essentially bagging
    bootstrap=True,
    min_samples_leaf=2,
    random_state=657677
)

single_clf_tree.fit(X_clf, y_clf)
bagged_clf_trees.fit(X_clf, y_clf)

In [ ]:
plot_classifier_boundary(
    single_clf_tree,
    X_clf,
    y_clf,
    title="Single classification tree"
)

plot_classifier_boundary(
    bagged_clf_trees,
    X_clf,
    y_clf,
    title="Bagged classification trees"
)

## Out-of-bag error

Each bootstrap sample leaves out some observations. For tree $b$, let

$$
O_b
$$

be the set of training indices that were **out of bag** for that tree.

For a fixed training point $x_n$, define

$$
B_n
=\{b: n\in O_b\},
$$

the set of trees for which observation $n$ was out of bag.

The out-of-bag score for $x_n$ is

$$
\hat s_{\text{OOB},n}(x_n)
=\frac{1}{|B_n|}\sum_{b\in B_n}\hat s_b(x_n).
$$

This is a prediction for $x_n$ **made only by trees that did not train on $x_n$**. So we can define

$$
\hat{y}_{\text{OOB},n} = a(\hat s_{\text{OOB},n}(x_n))
$$
and then use these OOB predictions to calculate error metrics. 

OOB error acts like an internal validation error. It is not exactly the same as a separate test set, but it is often a useful estimate of generalization performance since we're predicting on $x_n$ using a method that never saw $x_n$. 

In [ ]:
# OOB error for the regression example.

bagged_oob = RandomForestRegressor(
    n_estimators=500,
    max_features=1.0,
    bootstrap=True,
    oob_score=True, # This gives us OOB predictions
    min_samples_leaf=2,
    random_state=657677
)

bagged_oob.fit(X_reg, y)

In [ ]:
oob_pred = bagged_oob.oob_prediction_
oob_pred[:10]

In [ ]:
oob_mse = mean_squared_error(y, oob_pred)
train_mse = mean_squared_error(y, bagged_oob.predict(X_reg))

print("Training MSE:", round(train_mse, 4))
print("OOB MSE:", round(oob_mse, 4))
print("OOB R^2:", round(bagged_oob.oob_score_, 4))

## From bagging to random forests

Bagging uses bootstrap samples to make trees different. But if the same strong variables are always available at every split, many trees may still look similar (high $\rho$). Similar trees have highly correlated predictions. To solve this, instead of fitting a normal tree, random forests fit **Random Trees**. 

For a random tree, at each split, we only consider a random subset of the $D$ features.

Let

$$
M \leq D
$$

be the number of features considered at each split. Instead of searching over all features

$$
j=1,\dots,D,
$$

a random forest samples a subset

$$
\mathcal{J}_{\text{split}} \subset \{1,\dots,D\}
$$

with

$$
|\mathcal{J}_{\text{split}}|=M,
$$

and searches only over

$$
j\in \mathcal{J}_{\text{split}}.
$$

This makes the trees less correlated, which makes bagging more effective.

## Random forest algorithm

All together, then, the RF algorithm is: 

For $b=1,\dots,B$:

1. Draw a bootstrap sample $\mathcal{D}_N^{(b)}$ from the training data.
2. Grow a **random** tree on $\mathcal{D}_N^{(b)}$.
    + At each split, randomly select $M$ candidate features.
    + Choose the best split only among those $M$ features.
    + Grow the tree until a stopping rule is reached.
3. Bag the $B$ base-methods. 

In [ ]:
X_sim, y_sim = make_regression(
    n_samples=800,
    n_features=100,
    n_informative=100,
    noise=30.0,
    effective_rank=4,
    tail_strength=0.05,
    random_state=None
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_sim,
    y_sim,
    test_size=0.35,
    random_state=657677
)

In [ ]:
# Single deep tree
single_tree = DecisionTreeRegressor(
    max_depth=None,
    min_samples_leaf=2,
    random_state=657677
)

# bagging
# Counter-intuitively: If None or 1.0, then max_features=n_features, since its not int(1) but 100% frac
bag_like = RandomForestRegressor(
    n_estimators=3000,
    max_features=1.0,
    bootstrap=True,
    min_samples_leaf=2,
    random_state=657677
)


# RF
rf_sqrt = RandomForestRegressor(
    n_estimators=3000,
    max_features="sqrt",
    bootstrap=True,
    min_samples_leaf=2,
    random_state=657677
)

single_tree.fit(X_train, y_train)
bag_like.fit(X_train, y_train)
rf_sqrt.fit(X_train, y_train)

In [ ]:
def tree_prediction_summary(model, X_eval):
    preds = np.column_stack([tree.predict(X_eval) for tree in model.estimators_])
    corr = np.corrcoef(preds.T)
    off_diag = corr[np.triu_indices_from(corr, k=1)]
    return {
        "avg_pairwise_tree_corr": np.nanmean(off_diag),
        "avg_tree_pred_sd": preds.std(axis=1).mean(),
        "ensemble_pred_sd": preds.mean(axis=1).std(),
    }

summary = pd.DataFrame({
    "bagged trees, all features": tree_prediction_summary(bag_like, X_test),
    "random forest, sqrt features": tree_prediction_summary(rf_sqrt, X_test),
}).T

summary["test_RMSE"] = [
    np.sqrt(mean_squared_error(y_test, bag_like.predict(X_test))),
    np.sqrt(mean_squared_error(y_test, rf_sqrt.predict(X_test))),
]

print("Single tree test RMSE:")
print(np.sqrt(mean_squared_error(y_test, single_tree.predict(X_test))))

display(summary.round(4))

The random forest version often has lower pairwise correlation between trees because each split is forced to consider only a subset of features. This does not guarantee lower test error in every dataset, but it targets the variance formula:

$$
\operatorname{Var}(\bar Z_B)
=\rho\sigma^2+
\frac{1-\rho}{B}\sigma^2.
$$

Random forests try to make $\rho$ smaller without making each tree too weak.

Random forests are not guaranteed to improve on bagging. Feature subsampling reduces correlation between trees, but it also makes each tree weaker (higher bias) because each split sees only a subset of the variables. If there are only a few predictors, or if most of the signal is in one predictor, then too low `max_features` can be too aggressive and can perform worse than bagging.

## How many trees?

The number of trees is controlled by

$$
B.
$$

As $B$ increases, the random forest prediction stabilizes. More trees do not cause overfitting in the same way deeper individual trees can. Instead, more trees reduce Monte Carlo noise from the randomization.

The tradeoff is computation: more trees take more time and memory.

In [ ]:
# OOB error as a function of the number of trees.

n_tree_values = [10, 25, 50, 100, 200, 400]
oob_mse_values = []
test_mse_values = []

for B in n_tree_values:
    rf = RandomForestRegressor(
        n_estimators=B,
        max_features="sqrt",
        bootstrap=True,
        oob_score=True,
        min_samples_leaf=2,
        random_state=657677
    )
    rf.fit(X_train, y_train)

    oob_pred = rf.oob_prediction_
    oob_mse_values.append(mean_squared_error(y_train, oob_pred))
    test_mse_values.append(mean_squared_error(y_test, rf.predict(X_test)))

plt.figure(figsize=(7, 4))
plt.plot(n_tree_values, oob_mse_values, marker="o", label="OOB MSE")
plt.plot(n_tree_values, test_mse_values, marker="o", label="test MSE")
plt.xlabel("number of trees")
plt.ylabel("MSE")
plt.title("OOB and test error usually stabilize as the forest grows")
plt.legend()
plt.tight_layout()
plt.show()

## Real regression example: diabetes data

We now compare a single tree, bagged trees, and a random forest on a real regression dataset.

The response is a quantitative disease-progression outcome. The predictors are baseline measurements.

In [ ]:
# Load diabetes data.

diabetes = load_diabetes(as_frame=True)
X_diab = diabetes.data
y_diab = diabetes.target

X_train, X_test, y_train, y_test = train_test_split(
    X_diab,
    y_diab,
    test_size=0.30,
    random_state=657677
)

models_reg = {
    "single tree": DecisionTreeRegressor(
        min_samples_leaf=5,
        random_state=657677
    ),
    "bagged trees": RandomForestRegressor(
        n_estimators=500,
        max_features=1.0,
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
    "random forest": RandomForestRegressor(
        n_estimators=500,
        max_features="sqrt",
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
}

rows = []

for name, model in models_reg.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    row = {
        "model": name,
        "train_RMSE": np.sqrt(mean_squared_error(y_train, pred_train)),
        "test_RMSE": np.sqrt(mean_squared_error(y_test, pred_test)),
    }

    if hasattr(model, "oob_prediction_"):
        row["OOB_RMSE"] = np.sqrt(mean_squared_error(y_train, model.oob_prediction_))
    else:
        row["OOB_RMSE"] = np.nan

    rows.append(row)

results_reg = pd.DataFrame(rows)
display(results_reg.round(3))

In [ ]:
# Feature importance for the random forest regression model.

rf_reg = models_reg["random forest"]

importance_reg = pd.DataFrame({
    "feature": X_diab.columns,
    "impurity_importance": rf_reg.feature_importances_,
}).sort_values("impurity_importance", ascending=False)

display(importance_reg.round(4))

plt.figure(figsize=(7, 4))
plt.barh(importance_reg["feature"], importance_reg["impurity_importance"])
plt.gca().invert_yaxis()
plt.xlabel("impurity-based importance")
plt.title("Random forest feature importance, diabetes regression")
plt.tight_layout()
plt.show()

## Feature importance

Random forests often report feature importance scores. The usual impurity-based score works like this:

- every time a feature is used in a split, measure how much the split reduces RSS or impurity;
- add those reductions across all splits in all trees;
- normalize so that the scores sum to one.

These scores are useful, but they should be interpreted cautiously. They can be biased toward variables with many possible split points, and correlated features can split importance between each other.

A more model-agnostic alternative is **permutation importance**: randomly permute one feature in the test set and measure how much performance gets worse.

In [ ]:
# Permutation importance on the test set.

perm_reg = permutation_importance(
    rf_reg,
    X_test,
    y_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=15,
    random_state=657677
)

perm_importance_reg = pd.DataFrame({
    "feature": X_diab.columns,
    "permutation_importance": perm_reg.importances_mean,
    "sd": perm_reg.importances_std,
}).sort_values("permutation_importance", ascending=False)

display(perm_importance_reg.round(4))

## Real classification example: breast cancer data

Now we use a binary classification dataset. We compare a single classification tree, bagged trees, and a random forest.

In [ ]:
# Load breast cancer data.

cancer = load_breast_cancer(as_frame=True)
X_cancer = cancer.data
y_cancer = cancer.target
class_names = cancer.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X_cancer,
    y_cancer,
    test_size=0.30,
    stratify=y_cancer,
    random_state=657677
)

models_clf = {
    "single tree": DecisionTreeClassifier(
        min_samples_leaf=5,
        random_state=657677
    ),
    "bagged trees": RandomForestClassifier(
        n_estimators=500,
        max_features=1.0,
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
    "random forest": RandomForestClassifier(
        n_estimators=500,
        max_features="sqrt",
        bootstrap=True,
        min_samples_leaf=5,
        oob_score=True,
        random_state=657677
    ),
}

rows = []

for name, model in models_clf.items():
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)

    row = {
        "model": name,
        "train_accuracy": accuracy_score(y_train, pred_train),
        "test_accuracy": accuracy_score(y_test, pred_test),
    }

    if hasattr(model, "oob_score_"):
        row["OOB_accuracy"] = model.oob_score_
    else:
        row["OOB_accuracy"] = np.nan

    rows.append(row)

results_clf = pd.DataFrame(rows)
display(results_clf.round(4))

In [ ]:
# Confusion matrix for the random forest classifier.

rf_clf = models_clf["random forest"]
y_pred = rf_clf.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=class_names
)
plt.title("Random forest confusion matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance for the classification forest.

importance_clf = pd.DataFrame({
    "feature": X_cancer.columns,
    "impurity_importance": rf_clf.feature_importances_,
}).sort_values("impurity_importance", ascending=False)

display(importance_clf.head(15).round(4))

plt.figure(figsize=(7, 5))
plt.barh(importance_clf.head(15)["feature"], importance_clf.head(15)["impurity_importance"])
plt.gca().invert_yaxis()
plt.xlabel("impurity-based importance")
plt.title("Top random forest feature importances, breast cancer data")
plt.tight_layout()
plt.show()

## Main tuning parameters

Important random forest tuning parameters include:

**Number of trees**

$$
B.
$$

More trees reduce Monte Carlo noise, but increase computation.

**Number of features considered at each split**

$$
M.
$$

Smaller $M$ makes trees less correlated, but each individual split may be weaker.

**Tree size**

Controlled through parameters such as maximum depth, minimum samples per leaf, or minimum samples needed to split. Deeper trees have lower bias but higher variance before averaging.

## Practical notes

### Scaling

Random forests are usually not sensitive to monotone rescaling of individual features. A split like

$$
x_j \leq t
$$

has an equivalent split after changing units. Standardization is therefore not usually required.

### Categorical variables

Conceptually, trees can split categorical variables by grouping categories. Implementation details vary. In many Python workflows, categorical variables are one-hot encoded before using `sklearn` random forests.

### Missing values

Some modern tree implementations handle missing values internally. Depending on the software and version, `sklearn` support varies by tree estimator and criterion. A safe general workflow is to use an imputer inside a pipeline when missing values are present.

### Interpretation

A single tree can be drawn and inspected. A random forest is harder to interpret because it averages many trees. Feature importance and partial dependence plots can help summarize the fitted forest, but they are not the same as reading off a simple set of rules.

### Extrapolation

Trees and random forests are piecewise constant or averages of piecewise constant functions. They are usually poor at extrapolating outside the range of the training data.

## Review Questions

See: @sec-rf-questions.